In [112]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import SelectFromModel

In [4]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('fraudTrain.csv')

# Display the first few rows of the dataframe
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [114]:
df = df.drop(columns=['Unnamed: 0', 'trans_num'])

In [115]:
# Preprocessing
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])
df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365

In [116]:
# Feature engineering
df['trans_year'] = df['trans_date_trans_time'].dt.year
df['trans_month'] = df['trans_date_trans_time'].dt.month
df['trans_day'] = df['trans_date_trans_time'].dt.day
df['trans_hour'] = df['trans_date_trans_time'].dt.hour
df['distance'] = np.sqrt((df['lat'] - df['merch_lat'])**2 + (df['long'] - df['merch_long'])**2)

In [117]:
# Frequency Encoding for categorical features
for col in ['first', 'last', 'street', 'city', 'category', 'state', 'gender', 'merchant', 'job']:
    freq_encoding = df[col].value_counts().to_dict()
    df[col] = df[col].map(freq_encoding)

In [118]:
from sklearn.preprocessing import StandardScaler

numeric_features = ['amt', 'city_pop' , 'unix_time'] 

# Normalize numerical features
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

In [119]:
df = df.drop(columns=['trans_date_trans_time', 'dob'])

In [120]:
y = df[['is_fraud']]

In [121]:
# Define features and target
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

In [6]:
# Convert all float columns to int
float_columns = df.select_dtypes(include=['float']).columns
df[float_columns] = df[float_columns].astype(int)

In [156]:
# Handle class imbalance
from imblearn.under_sampling import RandomUnderSampler

# Undersampling
undersampler = RandomUnderSampler(sampling_strategy='auto')  # auto balances all classes
X_res, y_res = undersampler.fit_resample(X, y)

In [157]:
# Convert all columns to numeric, coercing errors (non-convertible values become NaN)
X_res = X_res.apply(pd.to_numeric, errors='coerce')

# Optionally, handle NaN values (e.g., impute or drop them)
X_res = X_res.fillna(0)  # Or use other imputation strategies

In [166]:
import warnings
warnings.filterwarnings("ignore")

In [167]:
# Feature selection using RandomForest
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_res, y_res)
selector = SelectFromModel(rf, prefit=True, threshold='median')
X_selected = selector.transform(X_res)

In [168]:
# Split the data
X_train, X_val, y_train, y_val = train_test_split(X_selected, y_res, test_size=0.2, random_state=42)

# Standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [169]:
# Define models and their hyperparameters for GridSearchCV
models = {
    'Logistic Regression': (LogisticRegression(), {'C': [0.01, 0.1, 1, 10, 100]}),
    'Decision Tree': (DecisionTreeClassifier(), {'max_depth': [5, 10, 15, 20]}),
    'Random Forest': (RandomForestClassifier(), {'n_estimators': [50, 100, 200]}),
    'Gradient Boosting': (GradientBoostingClassifier(), {'n_estimators': [50, 100, 200]}),
    'SVM': (SVC(probability=True), {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}),
    'KNN': (KNeighborsClassifier(), {'n_neighbors': [3, 5, 7]}),
    'Naive Bayes': (GaussianNB(), {})
}


In [170]:
# Cross-validation and GridSearch
cv_results = {}
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, (model, params) in models.items():
    grid_search = GridSearchCV(model, params, scoring='f1', cv=kf)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_val)
    y_proba = best_model.predict_proba(X_val)[:, 1]
    
    print(f'{name} Best Parameters: {grid_search.best_params_}')
    print(f'{name} Validation Performance:')
    print(f'Accuracy: {accuracy_score(y_val, y_pred)}')
    print(f'Precision: {precision_score(y_val, y_pred)}')
    print(f'Recall: {recall_score(y_val, y_pred)}')
    print(f'F1 Score: {f1_score(y_val, y_pred)}')
    print(f'ROC AUC: {roc_auc_score(y_val, y_proba)}')
    print('---')
    
    cv_results[name] = best_model


Logistic Regression Best Parameters: {'C': 10}
Logistic Regression Validation Performance:
Accuracy: 0.8478188478188479
Precision: 0.9240816326530612
Recall: 0.7566844919786097
F1 Score: 0.8320470415288497
ROC AUC: 0.8824443151212346
---
Decision Tree Best Parameters: {'max_depth': 10}
Decision Tree Validation Performance:
Accuracy: 0.967032967032967
Precision: 0.9691067830758898
Recall: 0.964572192513369
F1 Score: 0.9668341708542715
ROC AUC: 0.9827367561007633
---
Random Forest Best Parameters: {'n_estimators': 100}
Random Forest Validation Performance:
Accuracy: 0.9716949716949717
Precision: 0.9725385130609511
Recall: 0.9705882352941176
F1 Score: 0.9715623954499832
ROC AUC: 0.9962465712592572
---
Gradient Boosting Best Parameters: {'n_estimators': 200}
Gradient Boosting Validation Performance:
Accuracy: 0.9693639693639694
Precision: 0.9724091520861373
Recall: 0.9659090909090909
F1 Score: 0.9691482226693495
ROC AUC: 0.9955288865863049
---
SVM Best Parameters: {'C': 10, 'kernel': 'rbf'

In [136]:
# Load the test dataset
test_data_path = 'fraudTest.csv'
test_data = pd.read_csv(test_data_path)

In [137]:
# Preprocess the test dataset similarly
test_data['trans_date_trans_time'] = pd.to_datetime(test_data['trans_date_trans_time'])
test_data['dob'] = pd.to_datetime(test_data['dob'])
test_data['age'] = (test_data['trans_date_trans_time'] - test_data['dob']).dt.days // 365

test_data['trans_year'] = test_data['trans_date_trans_time'].dt.year
test_data['trans_month'] = test_data['trans_date_trans_time'].dt.month
test_data['trans_day'] = test_data['trans_date_trans_time'].dt.day
test_data['trans_hour'] = test_data['trans_date_trans_time'].dt.hour
test_data['distance'] = np.sqrt((test_data['lat'] - test_data['merch_lat'])**2 + (test_data['long'] - test_data['merch_long'])**2)

In [138]:
# Frequency Encoding for categorical features
for col in ['first', 'last', 'street', 'city', 'category', 'state', 'gender', 'merchant', 'job']:
    freq_encoding = test_data[col].value_counts().to_dict()
    test_data[col] = test_data[col].map(freq_encoding)

In [139]:
from sklearn.preprocessing import StandardScaler

numeric_features = ['amt', 'city_pop' , 'unix_time'] 

# Normalize numerical features
scaler = StandardScaler()
test_data[numeric_features] = scaler.fit_transform(test_data[numeric_features])

In [140]:
test_data = test_data.drop(columns=['trans_date_trans_time', 'dob', 'Unnamed: 0', 'trans_num'])

In [141]:
# Convert all float columns to int
float_columns = test_data.select_dtypes(include=['float']).columns
test_data[float_columns] = test_data[float_columns].astype(int)

In [143]:
test_data.shape


(555719, 25)

In [144]:
df.shape

(1296675, 25)

In [171]:
X_test = test_data.drop('is_fraud', axis=1)
y_test = test_data['is_fraud']

In [172]:
# Convert all columns to numeric, coercing errors (non-convertible values become NaN)
X_test = X_test.apply(pd.to_numeric, errors='coerce')

# Handle NaN values, using the same strategy as for the training data
X_test = X_test.fillna(0) 

In [173]:
# Ensure X_test has the same columns as X_res
X_test = X_test[X_res.columns]

In [174]:
import pandas as pd

# Convert X_res and X_test to DataFrames if they aren't already
X_res = pd.DataFrame(X_res)
X_test = pd.DataFrame(X_test)


In [175]:
X_test_selected = selector.transform(X_test)
X_test_scaled = scaler.transform(X_test_selected)

In [176]:
# Evaluate on the test dataset
for name, model in cv_results.items():
    y_pred_test = model.predict(X_test_scaled)
    y_proba_test = model.predict_proba(X_test_scaled)[:, 1]

    print(f'{name} Test Performance:')
    print(f'Accuracy: {accuracy_score(y_test, y_pred_test)}')
    print(f'Precision: {precision_score(y_test, y_pred_test)}')
    print(f'Recall: {recall_score(y_test, y_pred_test)}')
    print(f'F1 Score: {f1_score(y_test, y_pred_test)}')
    print(f'ROC AUC: {roc_auc_score(y_test, y_proba_test)}')
    print('---')

Logistic Regression Test Performance:
Accuracy: 0.9661159686820138
Precision: 0.08092630732908022
Recall: 0.7510489510489511
F1 Score: 0.14610919644476691
ROC AUC: 0.8849647621036811
---
Decision Tree Test Performance:
Accuracy: 0.9141814478180519
Precision: 0.0012265101406106268
Recall: 0.026107226107226107
F1 Score: 0.0023429491872895007
ROC AUC: 0.34726470093810324
---
Random Forest Test Performance:
Accuracy: 0.8614515609507682
Precision: 0.011716354621958379
Recall: 0.41864801864801865
F1 Score: 0.022794770910013962
ROC AUC: 0.8143872342893611
---
Gradient Boosting Test Performance:
Accuracy: 0.9948823056256849
Precision: 0.3638488507985976
Recall: 0.4354312354312354
F1 Score: 0.3964346349745331
ROC AUC: 0.8756355250424698
---
SVM Test Performance:
Accuracy: 0.7698297160975242
Precision: 0.0112695760307776
Recall: 0.675990675990676
F1 Score: 0.0221695589022246
ROC AUC: 0.8041367503457486
---
KNN Test Performance:
Accuracy: 0.8294911636996396
Precision: 0.013602941176470588
Recall: